## Notebook for testing our pipeline for the Generative Recommender

In [115]:
# Imports here
from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
import torch
import numpy as np
import pickle
import os
import torch, numpy as np, random

# This is ONLY for our tests! Do not use for our full model
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### First, get the feature strings from DataHandler

In [116]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [117]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

The embeddings have the shape: (105542, 384)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [ ]:
input_dim = embeddings.shape[1]
latent_dim = 32
hidden_dim = 256
codebook_size = 512
num_quantizers = 4

set_seed(42)
rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# trained_model = ...

print('Generating semantic IDs')

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()
semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

print(f'semantic_ids has shape: {semantic_ids.shape}')
print('printing the first semantic ID (untrained RQ-VAE)')
print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('item_2_semantic.pkl') and os.path.exists('semantic_2_item.pkl'):
    print('Loading existing hashmaps')

    with open('item_2_semantic.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('semantic_2_item.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:

    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')

    with open('item_2_semantic.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('semantic_2_item.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Generating semantic IDs
semantic_ids has shape: (105542, 4)
printing the first semantic ID (untrained RQ-VAE)
[ 28 450 349 425]
Loading existing hashmaps
Loaded the hashmaps
